# 02. Lojas de moda: blocagem quando as unidades são heterogêneas

**Setor:** varejo físico de moda. **Decisão:** um novo layout de vitrine
aumenta as vendas? As unidades são **120 lojas**, e elas são muito diferentes
entre si: uma loja `G` vende muito mais que uma `PP`, independentemente do
layout. Essa variação entre tamanhos, se não for controlada, "afoga" o efeito.

A resposta é **blocar por tamanho de loja** (`PP`, `P`, `M`, `G`): randomizar
dentro de cada bloco e combinar por média ponderada. Base teórica:
[II. Desenhos](../../../docs/pt-br/teoria/02-desenhos.md) (tópico 5) e
[III. Estimação](../../../docs/pt-br/teoria/03-estimacao.md) (tópico 9).


In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd


def _find_data():
    """Locate examples/use_cases/data whether run from the notebook or the root."""
    for base in [Path.cwd(), *Path.cwd().parents]:
        for cand in (base / "data", base / "examples" / "use_cases" / "data"):
            if (cand / "ecommerce_checkout.csv").exists():
                return cand
    raise FileNotFoundError("Could not locate examples/use_cases/data")


DATA = _find_data()

df = pd.read_csv(DATA / "fashion_stores.csv")
# Baseline sales differ strongly by size: this is the between-block variation.
print(df.groupby("store_size")["sales_y0"].mean().round(2))
df.head()


## 1. Desenho blocado e equilíbrio

Randomizamos dentro de cada tamanho com `BlockedCRD(block_col="store_size")`.
Logo após o sorteio (antes de anexar o resultado), checamos o **equilíbrio** das
covariáveis com `check_balance`: a randomização deve deixar tratamento e
controle parecidos em `foot_traffic_pre`. Avaliamos pela **magnitude da SMD**
(`|SMD| < 0,1` é bom), não por p-valor.


In [ ]:
from skxperiments.core.assignment import BlockedAssignment
from skxperiments.design.blocked_crd import BlockedCRD
from skxperiments.design.balance import check_balance

design = BlockedCRD(block_col="store_size", p=0.5, seed=202)
assignment = design.randomize(df[["region", "store_size", "foot_traffic_pre"]].copy())

# Balance is checked on the pre-treatment covariates only.
check_balance(assignment)[["covariate", "smd"]].round(3)


In [ ]:
from skxperiments.diagnostics import BalanceReport
from skxperiments.reporting import plot_balance

report = BalanceReport().run(assignment)
ax = plot_balance(report)
ax.set_title("Covariate balance (Love plot)")
ax.figure


## 2. Estimativa blocada vs. ignorar os blocos

Anexamos o resultado observado e estimamos com `BlockedDifferenceInMeans` (média
dos efeitos por bloco, ponderada por tamanho) e `NeymanCI` (variância
estratificada). Para deixar o ganho explícito, comparamos com a análise
**ingênua** que trata tudo como um `CRD` único, ignorando o tamanho da loja: o
erro padrão fica bem maior, porque a variação entre tamanhos entra no ruído.


In [ ]:
from skxperiments.core.assignment import CRDAssignment
from skxperiments.design.crd import CRD
from skxperiments.estimators.blocked_difference_in_means import BlockedDifferenceInMeans
from skxperiments.estimators.difference_in_means import DifferenceInMeans
from skxperiments.inference import NeymanCI

t = assignment.data_[assignment.treatment_col_].to_numpy()
data = assignment.data_.copy()
data["sales"] = np.where(t == 1, df["sales_y1"].to_numpy(), df["sales_y0"].to_numpy())

blocked_a = BlockedAssignment(
    data=data, treatment_col=assignment.treatment_col_, design=design,
    block_col=assignment.block_col_, block_sizes=assignment.block_sizes_, seed=202,
)
blocked_est = BlockedDifferenceInMeans(outcome_col="sales").fit(blocked_a)
blocked = NeymanCI(estimator=BlockedDifferenceInMeans(outcome_col="sales")).fit(blocked_a).estimate()

# Naive: same data, but analyzed as a single CRD (blocks ignored).
naive_a = CRDAssignment(data=data, treatment_col="treatment", design=CRD(p=0.5, seed=202), seed=202)
naive = NeymanCI(estimator=DifferenceInMeans(outcome_col="sales")).fit(naive_a).estimate()

print("per-block ATE:", {k: round(v, 3) for k, v in blocked_est.block_ates_.items()})
pd.DataFrame([
    {"analysis": "Blocked (correct)", "ATE": blocked.ate, "SE": blocked.se,
     "CI_low": blocked.ci[0], "CI_high": blocked.ci[1], "p": blocked.p_value},
    {"analysis": "Naive CRD (blocks ignored)", "ATE": naive.ate, "SE": naive.se,
     "CI_low": naive.ci[0], "CI_high": naive.ci[1], "p": naive.p_value},
]).round(3)


In [ ]:
from skxperiments.reporting import plot_effect

ax = plot_effect(blocked)
ax.set_title("Sales lift, blocked estimate (95% CI)")
ax.figure


## Decisão

O efeito verdadeiro é `+0,40` (mil em vendas diárias), constante entre tamanhos.
A estimativa blocada fica perto disso e o intervalo exclui zero. Compare os erros
padrão: a análise blocada é **bem mais precisa** que a ingênua, porque removeu do
erro a variação entre tamanhos de loja. Mesma coleta de dados, decisão mais
firme, só por respeitar a estrutura do desenho.

Próximo passo: [03. Campanha fatorial](03_fintech_factorial.ipynb).
